In [ ]:
import numpy as np
import pandas as pd
import xarray as xr
from pathlib import Path

PROJECT_ROOT = Path("..").resolve()
BRONZE_PATH = PROJECT_ROOT / "data/gulf_stream_pigment_influencers_20241001_20251231_v1/bronze"
SILVER_PATH = PROJECT_ROOT / "data/gulf_stream_pigment_influencers_20241001_20251231_v1/silver"
GOLD_PATH = PROJECT_ROOT / "data/gulf_stream_pigment_influencers_20241001_20251231_v1/gold"

pd.set_option("display.max_rows", 1000)

### Bailey and Werdell (2006) validation procedure

For each satellite and in situ pair:

- Use native-resolution satellite data instead of reduced-resolution data.
- Require the in situ measurement to be within 3 hours of the satellite overpass.
- Use a 5 x 5 pixel box centered on the in situ location.
- Require each satellite record to be unique and to share no pixels with another validation record.
- Exclude certain flags (see validation/quality.py).
- Require at least 50% of the 25 pixels to remain valid.
- For coastal water, require at least 50% of non-land pixels and at least 5 valid pixels.
- Remove values outside the initial mean plus or minus 1.5 standard deviations.
- Compute the coefficient of variation from the filtered mean and standard deviation.
- Reject the box when the median coefficient of variation exceeds 0.15.

In [ ]:
from eddy_tracking.validation.seabass import read_hplc_dir

hplc_df = read_hplc_dir("../data/sdp_validation/pvst_bats_hplc")
hplc_df["datetime"] = hplc_df["datetime"].dt.tz_localize("UTC")

# depth is sample depth below the water surface, in meters
# water_depth is distance from water surface to sea floor
# Anything < 200 m is coastal water; > 1000 m is open ocean
meta_cols = ["depth", "water_depth", "datetime", "lon", "lat"]

# SeaBASS column -> SDP model pigment
seabass_to_sdp = {
    "allo": "Allo",
    "but-fuco": "ButFuco",
    "chl_c1c2": "chl c1+c2",
    "chl_c3": "chl c3",
    "dv_chl_a": "DV chla",
    "fuco": "Fuco",
    "hex-fuco": "HexFuco",
    "mv_chl_b": "MV chlb",
    "neo": "Neo",
    "perid": "Perid",
    "tot_chl_a": "T chla",
    "viola": "Viola",
    "zea": "Zea",
}
pigment_cols = list(seabass_to_sdp.keys())

keep_cols = set(meta_cols) | set(pigment_cols)
hplc_df = hplc_df.loc[:, hplc_df.columns.isin(keep_cols)]
hplc_df.head()

In [ ]:
# This departs from Bailey and Werdell: find the closest valid PACE pixel first, build the native 5x5 box from its scan_line and pixel, then apply QA inside that box.
# Disqualifying rows are dropped before the mean, so the satellite value is the filtered arithmetic mean of the survivors.

from datetime import timedelta

import earthaccess

from eddy_tracking.validation.matchup import list_pace_l2_matchups, list_sss_matchups, list_sst_matchups
from eddy_tracking.utils.authentication import login_earthdata
from eddy_tracking.preprocess.pace import read_multiple_pace_l2
from eddy_tracking.preprocess.sss import read_multiple_sss
from eddy_tracking.preprocess.sst import read_multiple_sst
from eddy_tracking.validation.quality import apply_l2_quality_flags
from eddy_tracking.utils.geography import get_5x5_pace_l2_matchups
from eddy_tracking.packages.sdp.prediction import run_sdp_on_pace_l2

def filtered_arithmetic_mean(values: pd.Series) -> float:
    values = pd.to_numeric(values, errors="coerce").replace([np.inf, -np.inf], np.nan).dropna()
    if values.empty:
        return float("nan")

    mean = values.mean()
    std = values.std()
    if pd.isna(std) or std == 0:
        return float(mean)

    filtered_values = values[(values > mean - 1.5 * std) & (values < mean + 1.5 * std)]
    return float(filtered_values.mean())

login_earthdata()
pace_l2_download_dir = Path("../data/sdp_validation/pace_l2")
sss_download_dir = Path("../data/sdp_validation/sss")
sst_download_dir = Path("../data/sdp_validation/sst")
pigment_output_dir = PROJECT_ROOT / "data/sdp_validation/pigments"
pace_l2_download_dir.mkdir(parents=True, exist_ok=True)
sss_download_dir.mkdir(parents=True, exist_ok=True)
sst_download_dir.mkdir(parents=True, exist_ok=True)
pigment_output_dir.mkdir(parents=True, exist_ok=True)

unique_measurements = hplc_df[["lon", "lat", "datetime"]].drop_duplicates().reset_index(drop=True)
window = timedelta(hours=3)

saved_ct = 0
for lon, lat, dttm in unique_measurements.itertuples(index=False, name=None):
    # Download data
    pace_l2_granules = list_pace_l2_matchups(lon, lat, dttm, window)
    downloaded_pace_l2_fps = earthaccess.download(
        granules=pace_l2_granules,
        local_path=pace_l2_download_dir,
    )
    sss_granules = list_sss_matchups(lon, lat, dttm, window)
    downloaded_sss_fps = earthaccess.download(
        granules=sss_granules,
        local_path=sss_download_dir,
    )
    sst_granules = list_sst_matchups(lon, lat, dttm, window)
    downloaded_sst_fps = earthaccess.download(
        granules=sst_granules,
        local_path=sst_download_dir,
    )

    # Read dataframes for the matched granules (lon, lat, dttm)
    pace_l2_matchup_df = read_multiple_pace_l2(downloaded_pace_l2_fps)
    sss_matchup_df = read_multiple_sss(downloaded_sss_fps)
    sst_matchup_df = read_multiple_sst(downloaded_sst_fps)

    # Limit PACE to 5x5 window
    pace_l2_matchup_df = get_5x5_pace_l2_matchups(pace_l2_matchup_df, lon, lat)

    # 13 of the 25 box pixels is the Bailey and Werdell 50% rule
    qa_df_len = len(apply_l2_quality_flags(pace_l2_matchup_df, to_nan=False))
    if qa_df_len < 13:
        print(
            "status: pace_sdp_prediction_skipped\n"
            "reason: insufficient_non_nan_points"
        )
        continue

    pace_l2_matchup_df = apply_l2_quality_flags(pace_l2_matchup_df)

    # Run SDP and cache results
    sdp = run_sdp_on_pace_l2(pace_l2_matchup_df, sst_matchup_df, sss_matchup_df)

    dttm_label = pd.Timestamp(dttm).strftime("%Y%m%dT%H%M%SZ")
    sdp_fp = pigment_output_dir / (
        f"sdp_lon_{lon:.6f}_lat_{lat:.6f}_dttm_{dttm_label}.parquet"
    )
    sdp.to_parquet(sdp_fp, index=False)
    saved_ct += 1
    print(
        f"sdp_parquet_path: {sdp_fp}\n"
        "status: saved"
    )


print(f"sdp_parquet_files_saved: {saved_ct}")

In [ ]:
# Read the saved SDP parquets and compare them to the in-situ HPLC values

import re

from eddy_tracking.utils.geography import calculate_dist_to_point


# Read the SDP output and attach context of in-situ lon/lat/dttm
sdp_pigment_cols = list(seabass_to_sdp.values())
sdp_fp_pattern = re.compile(r"sdp_lon_(-?[\d.]+)_lat_(-?[\d.]+)_dttm_(\d{8}T\d{6})Z\.parquet")

sdp_rows = []
for sdp_fp in sorted(pigment_output_dir.glob("sdp_lon_*.parquet")):
    lon_text, lat_text, dttm_text = sdp_fp_pattern.match(sdp_fp.name).groups()
    box = pd.read_parquet(sdp_fp)
    in_situ_lon = float(lon_text)
    in_situ_lat = float(lat_text)
    in_situ_datetime = pd.to_datetime(
        dttm_text,
        format="%Y%m%dT%H%M%S",
        utc=True,
    )
    distances = calculate_dist_to_point(
        box["longitude"],
        box["latitude"],
        in_situ_lon,
        in_situ_lat,
    )
    closest_pixel = box.loc[distances.idxmin()]
    row = {
        "in_situ_lon": in_situ_lon,
        "in_situ_lat": in_situ_lat,
        "in_situ_datetime": in_situ_datetime,
        "lon": float(closest_pixel["longitude"]),
        "lat": float(closest_pixel["latitude"]),
        "datetime": pd.Timestamp(closest_pixel["datetime"]),
        "datetime_start": pd.Timestamp(box["datetime"].min()),
        "datetime_end": pd.Timestamp(box["datetime"].max()),
        "box_pixels": len(box),
    }
    # One satellite value per pigment: the filtered arithmetic mean over the 5x5 box
    for pigment in sdp_pigment_cols:
        row[pigment] = filtered_arithmetic_mean(box[pigment])
    sdp_rows.append(row)

sdp_means = pd.DataFrame(sdp_rows)

# Take the average of all measurements at same (lon, lat, dttm) which are near surface
in_situ_means = (
    hplc_df.loc[hplc_df["depth"] <= 10]
    .rename(columns=seabass_to_sdp)
    .groupby(["lon", "lat", "datetime"], as_index=False)[sdp_pigment_cols]
    .mean()
    .rename(
        columns={
            "lon": "in_situ_lon",
            "lat": "in_situ_lat",
            "datetime": "in_situ_datetime",
        }
    )
)

# Join the HPLC pigments with SDP-predicted pigments
matchups = in_situ_means.merge(
    sdp_means,
    on=["in_situ_lon", "in_situ_lat", "in_situ_datetime"],
    suffixes=("_in_situ", "_sdp"),
)

pigment_frames = []

for pigment in sdp_pigment_cols:
    pigment_frame = pd.DataFrame(
        {
            "in_situ_lon": matchups["in_situ_lon"],
            "in_situ_lat": matchups["in_situ_lat"],
            "in_situ_datetime": matchups["in_situ_datetime"],
            "satellite_lon": matchups["lon"],
            "satellite_lat": matchups["lat"],
            "satellite_datetime": matchups["datetime"],
            "pigment": pigment,
            "in_situ": matchups[f"{pigment}_in_situ"],
            "sdp": matchups[f"{pigment}_sdp"],
        }
    )
    pigment_frames.append(pigment_frame)

comparison = pd.concat(pigment_frames, ignore_index=True)
comparison["difference"] = comparison["sdp"] - comparison["in_situ"]
comparison["percent_difference"] = 100 * comparison["difference"] / comparison["in_situ"]

summary = (
    comparison.groupby("pigment")
    .agg(
        n=("difference", "count"),
        in_situ_mean=("in_situ", "mean"),
        sdp_mean=("sdp", "mean"),
        bias=("difference", "mean"),
        mean_absolute_error=("difference", lambda values: values.abs().mean()),
        mean_percent_difference=("percent_difference", "mean"),
    )
    .sort_values("in_situ_mean", ascending=False)
)

print(
    f"sdp_boxes_read: {len(sdp_means)}\n"
    f"surface_hplc_matches: {len(matchups)}"
)
display(summary)

Subsequent Diagnostic Pigment Analysis to check PFTs derived from SDP pigments vs. PFTs derived from HPLC pigments + DPA (empirical ratios for PFTs by pigment)

- Write up a little bit of work we've investigated into
- We know cyclones have some enhancement in Northern Hemisphere; but can we try to quantify this on a broader level

- Focus on a few major regions; quantify (leaning toward) pigments
- Even if we get a negative result, either have an explanation for that, qualify it, or dive a little bit deeper like on pigments level